In [ ]:
# ==============================================================================
# Purpose:
#   Run pseudobulk DE + Hallmark GSEA (fgsea) at celltype_level3 using edgeR GLM.
#
# Summary:
#   Part A) Within each disease stage (HRL/DCIS/BC):
#     - Test NHB vs NHW (adjusting for age, diabetes, obesity)
#     - Run fgsea Hallmark on signed -log10(P) ranks
#     - Save: DEG_NHBvsNHW_celltypelevel3_EachDiseaseStage.csv
#             fgseaHallmark_NHBvsNHW_celltypelevel3_EachDiseaseStage.tsv
#
#   Part B) Within each race group (NHW/NHB):
#     - Test DCIS vs HRL, BC vs DCIS, BC vs HRL (adjusting covariates)
#     - Run fgsea Hallmark on signed -log10(P) ranks
#     - Save: DEG_BCvsDCISvsHRL_celltypelevel3_EachRaceGroup.csv
#             fgseaHallmark_BCvsDCISvsHRL_celltypelevel3_EachRaceGroup.tsv
#
# Notes:
#   - pb is an AggregateExpression Seurat object with counts in pb[["RNA"]]$counts
#     and per-sample metadata including dataset_name + celltype_level3.
#   - Requires adequate donors per cell type (threshold set below).
#   - Ranking metric uses PValue; sign comes from logFC.

# 081026, sensitivity analysis for DEG counts with batch as an additional covariate
# ==============================================================================

# Header

In [1]:
setwd('/n/data1/dfci/surgonc/mittendorf/lab/chelsea/iw/github')

In [2]:
source('codes/functions.R')

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t



Attaching package: ‘dplyr’


The following object is masked from ‘package:gridExtra’:

    combine


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘anndata’


The following object is masked from ‘package:SeuratObject’:

    Layers




In [3]:
library(edgeR)
library(fgsea)
library(tibble)

Loading required package: limma



# Load data

In [ ]:
dat = qs::qread('data/03_cleandata/ft_all.qs')
dat

Pseudobulk calculated in `07.4`

# Calc DEG

## Prep

In [4]:
pb = qs::qread('data/06_pb/pb_donor_celltypelevel3.qs')
pb

An object of class Seurat 
28202 features across 1341 samples within 5 assays 
Active assay: RNA (27879 features, 0 variable features)
 3 layers present: counts, data, scale.data
 4 other assays present: prediction.score.celltype.l1, prediction.score.celltype.l2, prediction.score.celltype.l3, impADT

In [5]:
meta_pt = read.csv('data/01_meta/meta_pt.csv', row.names = 1)

meta_pt$dataset_name <- gsub("_", "-", meta_pt$dataset_name) 

meta_pt

,dataset_name,batch,viability_pct,group,race,age,diabetes,obesity,race_stage
,<chr>,<int>,<int>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>
9-D,Pool137-7,1,95,DCIS,NHW,37.0,N,Y,NHW-DCIS
30-D,Pool137-12,1,92,DCIS,NHW,78.0,N,N,NHW-DCIS
36-D,Pool137-18,1,93,DCIS,NHB,39.0,N,Y,NHB-DCIS
119-D,Pool137-20,1,95,DCIS,NHB,80.0,N,Y,NHB-DCIS
28-D,Pool137-24,1,94,DCIS,NHW,53.0,N,N,NHW-DCIS
53-D,Pool137-27,1,96,DCIS,NHB,45.0,N,N,NHB-DCIS
77-D,Pool137-30,1,90,DCIS,NHW,45.0,N,N,NHW-DCIS
31-D,Pool138-6,2,93,DCIS,NHW,44.0,N,N,NHW-DCIS
6-H,Pool137-6,1,94,HRL,NHW,71.1,N,Y,NHW-HRL


In [6]:
table(meta_pt[,c('batch','race_stage')])

     race_stage
batch NHB-BC NHB-DCIS NHB-HRL NHW-BC NHW-DCIS NHW-HRL
    1      3        4       6      5        5       6
    2     10        0       4      7        1       4

In [7]:
meta_mg = merge(pb@meta.data, meta_pt,
                by='dataset_name'
               )
rownames(meta_mg) = meta_mg$orig.ident

meta_mg <- meta_mg %>%
  mutate(
    race     = factor(race, levels = c('NHW','NHB')),
    group    = factor(group, levels = c('HRL','DCIS','BC')),
    diabetes = factor(diabetes, levels = c('N','Y')),
    obesity  = factor(obesity, levels = c('N','Y')),

      batch = factor(batch, levels = c(1,2))
  )

head(meta_mg)

,dataset_name,orig.ident,celltype_level3,batch,viability_pct,group,race,age,diabetes,obesity,race_stage
,<chr>,<chr>,<chr>,<fct>,<int>,<fct>,<fct>,<dbl>,<fct>,<fct>,<chr>
Pool137-10_Naive B,Pool137-10,Pool137-10_Naive B,Naive B,1,88,HRL,NHW,62.3,N,N,NHW-HRL
Pool137-10_Mature B,Pool137-10,Pool137-10_Mature B,Mature B,1,88,HRL,NHW,62.3,N,N,NHW-HRL
Pool137-10_Memory B,Pool137-10,Pool137-10_Memory B,Memory B,1,88,HRL,NHW,62.3,N,N,NHW-HRL
Pool137-10_Plasmablast,Pool137-10,Pool137-10_Plasmablast,Plasmablast,1,88,HRL,NHW,62.3,N,N,NHW-HRL
Pool137-10_Naive CD4T,Pool137-10,Pool137-10_Naive CD4T,Naive CD4T,1,88,HRL,NHW,62.3,N,N,NHW-HRL
Pool137-10_Central Memory CD4T,Pool137-10,Pool137-10_Central Memory CD4T,Central Memory CD4T,1,88,HRL,NHW,62.3,N,N,NHW-HRL


In [8]:
celltypes_ordered = c(
    'Naive B','Mature B','Memory B','Plasmablast',
    'Naive CD4T','Central Memory CD4T','Effector Memory CD4T','Cytotoxic CD4T','Treg',
    'Naive CD8T','Central Memory CD8T','Effector Memory CD8T',
    'Double Negative T','γδ T','MAIT',
    'cDC1','cDC2','pDC',
    'Classical Monocyte','Intermediate Monocyte','Nonclassical Monocyte','Cytotoxic Monocyte',
    'CD56-dim NK','CD56-bright NK','Proliferating NK'
)

In [9]:
gmtfiles = c(
'h.all.v2024.1.Hs.symbols.gmt'
)

In [10]:
Cell_stress = c("UV RESPONSE UP", "UV RESPONSE DN", "UNFOLDED PROTEIN RESPONSE",
               "REACTIVE OXYGEN SPECIES PATHWAY", "P53 PATHWAY", "HYPOXIA", "G2M CHECKPOINT", "E2F TARGETS",
               "DNA REPAIR", "APOPTOSIS")
Inflammation = c("TNFA SIGNALING VIA NFKB", "INTERFERON GAMMA RESPONSE", "INTERFERON ALPHA RESPONSE",
                "IL6 JAK STAT3 SIGNALING", "IL2 STAT5 SIGNALING",
                "INFLAMMATORY RESPONSE", "COMPLEMENT", "ALLOGRAFT REJECTION")
Cell_cycle_metabolism = c("PROTEIN SECRETION", "PEROXISOME", 
                         "OXIDATIVE PHOSPHORYLATION", "MTORC1 SIGNALING",
                         "GLYCOLYSIS", "FATTY ACID METABOLISM", "ADIPOGENESIS")
Cancer = c("TGF BETA SIGNALING", "PI3K AKT MTOR SIGNALING", "MYC TARGETS V2", "MYC TARGETS V1", "MITOTIC SPINDLE",
          "KRAS SIGNALING UP", "KRAS SIGNALING DN", 
          "ESTROGEN RESPONSE EARLY", "ESTROGEN RESPONSE LATE", "EPITHELIAL MESENCHYMAL TRANSITION",
          "APICAL SURFACE", "APICAL JUNCTION", "ANDROGEN RESPONSE")

Miscellaneous = c("XENOBIOTIC METABOLISM", "WNT BETA CATENIN SIGNALING", "SPERMATOGENESIS",
                 "PANCREAS BETA CELLS", "NOTCH SIGNALING", "MYOGENESIS",
                  "HEME METABOLISM", "HEDGEHOG SIGNALING", 
                 "COAGULATION", "CHOLESTEROL HOMEOSTASIS", "BILE ACID METABOLISM",
                 "ANGIOGENESIS")

pathways_all = c(Cell_stress , Inflammation , Cell_cycle_metabolism , Cancer , Miscellaneous) # ordered

In [11]:
tmp_pt = unique(meta_mg[,c('dataset_name','group','race','diabetes','obesity')])

table(tmp_pt[,c('group','obesity')])

      obesity
group   N  Y
  HRL  12  8
  DCIS  5  5
  BC   14 11

In [12]:
tmp_sp = unique(meta_mg[,c('dataset_name','celltype_level3','group','race','diabetes','obesity')])

tmp_sp

,dataset_name,celltype_level3,group,race,diabetes,obesity
,<chr>,<chr>,<fct>,<fct>,<fct>,<fct>
Pool137-10_Naive B,Pool137-10,Naive B,HRL,NHW,N,N
Pool137-10_Mature B,Pool137-10,Mature B,HRL,NHW,N,N
Pool137-10_Memory B,Pool137-10,Memory B,HRL,NHW,N,N
Pool137-10_Plasmablast,Pool137-10,Plasmablast,HRL,NHW,N,N
Pool137-10_Naive CD4T,Pool137-10,Naive CD4T,HRL,NHW,N,N
Pool137-10_Central Memory CD4T,Pool137-10,Central Memory CD4T,HRL,NHW,N,N
Pool137-10_Cytotoxic CD4T,Pool137-10,Cytotoxic CD4T,HRL,NHW,N,N
Pool137-10_Effector Memory CD4T,Pool137-10,Effector Memory CD4T,HRL,NHW,N,N
Pool137-10_Treg,Pool137-10,Treg,HRL,NHW,N,N


In [13]:
table(tmp_sp$celltype_level3) %>% sort()


                 γδ T     Double Negative T                  MAIT 
                   40                    46                    52 
          Plasmablast        CD56-bright NK                  cDC1 
                   53                    54                    54 
                 cDC2            Naive CD8T      Proliferating NK 
                   54                    54                    54 
          CD56-dim NK   Central Memory CD4T   Central Memory CD8T 
                   55                    55                    55 
   Classical Monocyte        Cytotoxic CD4T    Cytotoxic Monocyte 
                   55                    55                    55 
 Effector Memory CD4T  Effector Memory CD8T Intermediate Monocyte 
                   55                    55                    55 
             Mature B              Memory B               Naive B 
                   55                    55                    55 
           Naive CD4T Nonclassical Monocyte                  

In [14]:
table(tmp_sp[,c('celltype_level3','group','race')])

, , race = NHW

                       group
celltype_level3         HRL DCIS BC
  CD56-bright NK         10    6 11
  CD56-dim NK            10    6 12
  cDC1                    9    6 12
  cDC2                   10    6 12
  Central Memory CD4T    10    6 12
  Central Memory CD8T    10    6 12
  Classical Monocyte     10    6 12
  Cytotoxic CD4T         10    6 12
  Cytotoxic Monocyte     10    6 12
  Double Negative T       9    4 11
  Effector Memory CD4T   10    6 12
  Effector Memory CD8T   10    6 12
  Intermediate Monocyte  10    6 12
  MAIT                    9    6 12
  Mature B               10    6 12
  Memory B               10    6 12
  Naive B                10    6 12
  Naive CD4T             10    6 12
  Naive CD8T             10    5 12
  Nonclassical Monocyte  10    6 12
  pDC                    10    6 12
  Plasmablast            10    6 12
  Proliferating NK       10    6 12
  Treg                   10    6 12
  γδ T                    6    6 10

, , race = NHB

  

In [15]:
table(tmp_sp[,c('celltype_level3','race','group')])

, , group = HRL

                       race
celltype_level3         NHW NHB
  CD56-bright NK         10  10
  CD56-dim NK            10  10
  cDC1                    9  10
  cDC2                   10  10
  Central Memory CD4T    10  10
  Central Memory CD8T    10  10
  Classical Monocyte     10  10
  Cytotoxic CD4T         10  10
  Cytotoxic Monocyte     10  10
  Double Negative T       9  10
  Effector Memory CD4T   10  10
  Effector Memory CD8T   10  10
  Intermediate Monocyte  10  10
  MAIT                    9  10
  Mature B               10  10
  Memory B               10  10
  Naive B                10  10
  Naive CD4T             10  10
  Naive CD8T             10  10
  Nonclassical Monocyte  10  10
  pDC                    10  10
  Plasmablast            10  10
  Proliferating NK       10  10
  Treg                   10  10
  γδ T                    6  10

, , group = DCIS

                       race
celltype_level3         NHW NHB
  CD56-bright NK          6   4
  CD56-dim N

## NHB vs NHW, in each disease state

In [16]:
results = data.frame()
dge_results_df = data.frame()

seed = 1024

for (celltype in celltypes_ordered){

    # need half or more donors to have this cell type
    if (table(tmp_sp[,'celltype_level3'])[[celltype]] < 27) {
        
        print(glue('No enough patients: {celltype}'))
        next
        
    } 
    
    ##--------------------------------- calc DEGs (edgeR) NHB vs NHW in each disease stage
    cell_ids <- colnames(pb)[pb$celltype_level3 == celltype]
    pb_sub <- pb[, cell_ids]
    
    counts = pb_sub[['RNA']]$counts
    meta = meta_mg %>% filter(celltype_level3 == celltype)
    
    keep_genes <- filterByExpr(counts, group = meta$group)
    counts <- counts[keep_genes, ]
    
    design <- model.matrix(~ race * group + age + diabetes + obesity + batch, # added batch
                           data = meta)
    
    y <- DGEList(counts = counts)
    y <- calcNormFactors(y)
    y <- estimateDisp(y, design)
    
    fit <- glmFit(y, design)

    colnames(design) <- make.names(colnames(design))
    contrasts <- list(
      HRL  = makeContrasts(raceNHB, levels = design),
      DCIS = makeContrasts(raceNHB + raceNHB.groupDCIS, levels = design),
      BC   = makeContrasts(raceNHB + raceNHB.groupBC, levels = design)
    )
    
  
    dge_results <- lapply(names(contrasts), function(grp) {
      lrt <- glmLRT(fit, contrast = contrasts[[grp]])
      res <- topTags(lrt, n = Inf)$table %>%
        rownames_to_column("gene") %>%
        mutate(group = grp)
      return(res)
    })


    ##--------------------------------- combine results and save DEG results
    dge_combined <- do.call(rbind, dge_results)
    rownames(dge_combined) <- NULL
    dge_combined$celltype = celltype

    dge_results_df = rbind(dge_results_df, dge_combined) 


    ##--------------------------------- fgsea
    ct = 0
    for (name in c('HRL','DCIS','BC')){

        #-------------- calc ranking for gsea
        ct = ct + 1
            
        de_sel = dge_results[[ct]] # HRL

        de_sel$fcSign <- sign(de_sel$logFC)
        
        # fix p-values equal to 0
        # first, check if p-values of 0 are detected in given cluster
        if (0 %in% de_sel$p_val_adj) { 
        
            # if so, count how many such values exist
            first_nonzero_p <- 0
            for (p in de_sel$p_val_adj) {
                if (p == 0) { 
                    first_nonzero_p <- first_nonzero_p + 1
                }
            }
        
            cat("\nNumber of (p_val_adj = 0) values:", first_nonzero_p, "\n")
        
            # create decreasing sequence from the last nonzero value 
            # (i.e. zero p-value closest to nonzero p-value)
            zero_values <- seq(first_nonzero_p, 1, by =-1) 
        
            # loop through all zero p-values starting with the one adjacent to nonzero value
            # for each p-value of 0, divide the previous non-zero value by the reduction
            # factor. this enable a valid nonzero p-value for each gene in each cluster
            reduction_factor <- 1.01
            for (zero_ind in zero_values) {
                de_sel$p_val_adj[zero_ind] <- (de_sel$p_val_adj[zero_ind+1]) / reduction_factor
                }
        }
        
        # add logP column (-log10 transformation of the p-value - not using FDR to avoid 23% ties)
        de_sel$logP <- -log10((de_sel$PValue )) 
        
        # add metric column (-log10 p-value with sign information)
        de_sel$metric <- de_sel$logP/de_sel$fcSign 
        
        ranks = setNames(de_sel$metric, de_sel$gene)
        
        #-------------- calc gsea
        for (gmt in gmtfiles){
            
            gmt_newname =  paste(strsplit(gmt, "\\.")[[1]][1:2], collapse = ".") # Combine the first two parts
        
            pathways = gmtPathways(glue("data/11_genesets/{gmt}"))
        
            set.seed(seed)
            fgseaRes <- fgsea(pathways=pathways, stats=ranks, nproc=1)
        
            fgseaRes$celltype = celltype
            fgseaRes$gmt = gmt_newname
            fgseaRes$subgroup = name
        
            results = rbind(results, fgseaRes)
        }
    }
    
    results$pathway <- gsub("^HALLMARK_", "", results$pathway)         # remove prefix
    results$pathway <- gsub("_", " ", results$pathway) 
    
}


# save DEG results
write.csv(dge_results_df, 
      'data/07RRE_pb_deg_fgsea/DEG_NHBvsNHW_celltypelevel3_EachDiseaseStage.csv',
      quote = F
     )


results$leadingEdge <- sapply(results$leadingEdge, function(x) paste(x, collapse = ", "))
write.table(results, 
            'data/07RRE_pb_deg_fgsea/fgseaHallmark_NHBvsNHW_celltypelevel3_EachDiseaseStage.tsv')

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%



Warning message in preparePathwaysAndStats(pathways, stats, minSize, maxSize, gseaParam, :
“There are ties in the preranked stats (0.01% of the list).
The order of those tied genes will be arbitrary, which may produce unexpected results.”


  |======================================================================| 100%

  |======================================================================| 100%



Warning message in preparePathwaysAndStats(pathways, stats, minSize, maxSize, gseaParam, :
“There are ties in the preranked stats (0.01% of the list).
The order of those tied genes will be arbitrary, which may produce unexpected results.”


  |======================================================================| 100%



...

  |======================================================================| 100%

  |======================================================================| 100%



Warning message in preparePathwaysAndStats(pathways, stats, minSize, maxSize, gseaParam, :
“There are ties in the preranked stats (0.01% of the list).
The order of those tied genes will be arbitrary, which may produce unexpected results.”


  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |=========================

Warning message in preparePathwaysAndStats(pathways, stats, minSize, maxSize, gseaParam, :
“There are ties in the preranked stats (0.02% of the list).
The order of those tied genes will be arbitrary, which may produce unexpected results.”


  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |=========================

In [17]:
print('Done!')

[1] "Done!"


## DCIS_vs_HRL, BC_vs_HRL, BC_vs_DCIS_NHB in each race group

In [18]:
results = data.frame()
dge_results_df = data.frame()

for (celltype in celltypes_ordered){

    # need half or more donors to have this cell type
    if (table(tmp_sp[,'celltype_level3'])[[celltype]] < 28) {
        
        print(glue('{celltype}'))
        next
        
    }
    
    ##--------------------------------- calc DEGs (edgeR) DCIS_vs_HRL, BC_vs_HRL, BC_vs_DCIS_NHB in each race group
    cell_ids <- colnames(pb)[pb$celltype_level3 == celltype]
    pb_sub <- pb[, cell_ids]
    
    counts = pb_sub[['RNA']]$counts
    meta = meta_mg %>% filter(celltype_level3 == celltype)
    
    keep_genes <- filterByExpr(counts, group = meta$group)
    counts <- counts[keep_genes, ]
    
    design <- model.matrix(~ race * group + age + diabetes + obesity + batch, # added batch
                           data = meta)
    colnames(design) <- make.names(colnames(design))

    y <- DGEList(counts = counts)
    y <- calcNormFactors(y)
    y <- estimateDisp(y, design)
    
    fit <- glmFit(y, design)

  # contrasts
  contrasts <- makeContrasts(
    # — in NHW subgroup (baseline race = NHW):  
    DCIS_vs_HRL_NHW = groupDCIS,                  
    BC_vs_DCIS_NHW  = groupBC - groupDCIS,         
    BC_vs_HRL_NHW   = groupBC,                     

    # — in NHB subgroup (race effect = everything with the NHB term):  
    DCIS_vs_HRL_NHB = groupDCIS + raceNHB.groupDCIS,                
    BC_vs_DCIS_NHB  = (groupBC + raceNHB.groupBC)                      
                      - (groupDCIS + raceNHB.groupDCIS),              
    BC_vs_HRL_NHB   = groupBC + raceNHB.groupBC,                      

    levels = design   # pass the numeric design matrix (now with valid names)
  )
 
  
  dge_results <- setNames(
    lapply(colnames(contrasts), function(grp) {
      lrt <- glmLRT(fit, contrast = contrasts[, grp])
      topTags(lrt, n = Inf)$table %>%
        rownames_to_column("gene") %>%
        mutate(subgroup = grp)
    }),
    colnames(contrasts)
  )


    ##--------------------------------- combine results and save DEG results
    dge_combined <- do.call(rbind, dge_results)
    rownames(dge_combined) <- NULL
    dge_combined$celltype = celltype

    dge_results_df = rbind(dge_results_df, dge_combined) 

    ##--------------------------------- fgsea
    for (name in names(dge_results)){
  
        #-------------- calc ranking for gsea
        de_sel = dge_results[[name]] # HRL
        de_sel$fcSign <- sign(de_sel$logFC)
        
        # fix p-values equal to 0
        # first, check if p-values of 0 are detected in given cluster
        if (0 %in% de_sel$p_val_adj) { 
        
            # if so, count how many such values exist
            first_nonzero_p <- 0
            for (p in de_sel$p_val_adj) {
                if (p == 0) { 
                    first_nonzero_p <- first_nonzero_p + 1
                }
            }
        
            cat("\nNumber of (p_val_adj = 0) values:", first_nonzero_p, "\n")
        
            # create decreasing sequence from the last nonzero value 
            # (i.e. zero p-value closest to nonzero p-value)
            zero_values <- seq(first_nonzero_p, 1, by =-1) 
        
            # loop through all zero p-values starting with the one adjacent to nonzero value
            # for each p-value of 0, divide the previous non-zero value by the reduction
            # factor. this enable a valid nonzero p-value for each gene in each cluster
            reduction_factor <- 1.01
            for (zero_ind in zero_values) {
                de_sel$p_val_adj[zero_ind] <- (de_sel$p_val_adj[zero_ind+1]) / reduction_factor
                }
        }
        
        # add logP column (-log10 transformation of the p-value - not using FDR to avoid 23% ties)
        de_sel$logP <- -log10((de_sel$PValue )) 
        
        # add metric column (-log10 p-value with sign information)
        de_sel$metric <- de_sel$logP/de_sel$fcSign 
        
        ranks = setNames(de_sel$metric, de_sel$gene)
        
        #-------------- calc gsea
        for (gmt in gmtfiles){
            
            gmt_newname =  paste(strsplit(gmt, "\\.")[[1]][1:2], collapse = ".") # Combine the first two parts
        
            pathways = gmtPathways(glue("data/11_genesets/{gmt}"))
        
            set.seed(seed)
            fgseaRes <- fgsea(pathways=pathways, stats=ranks, nproc=1)
        
            fgseaRes$celltype = celltype
            fgseaRes$gmt = gmt_newname
            fgseaRes$subgroup = name
        
            results = rbind(results, fgseaRes)
        }
    }
    
    results$pathway <- gsub("^HALLMARK_", "", results$pathway)         # remove prefix
    results$pathway <- gsub("_", " ", results$pathway) 
    
    # results_clean <- results %>%
    #   mutate(across(where(is.list), ~ sapply(., function(x) {
    #     if (is.null(x) || all(is.na(x))) return(NA_character_)
    #     paste(x, collapse = ",")
    #   }))) %>%
    #   mutate(across(everything(), ~ if (is.list(.)) as.character(.) else .))  # ensure no list columns

}

# save DEG results
write.csv(dge_results_df, 
          'data/07RRE_pb_deg_fgsea/DEG_BCvsDCISvsHRL_celltypelevel3_EachRaceGroup.csv',
          quote = F
         )

# save fgsea results
results$leadingEdge <- sapply(results$leadingEdge, function(x) paste(x, collapse = ", "))
write.table(results, 
            'data/07RRE_pb_deg_fgsea/fgseaHallmark_BCvsDCISvsHRL_celltypelevel3_EachRaceGroup.tsv')                            

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%



Warning message in preparePathwaysAndStats(pathways, stats, minSize, maxSize, gseaParam, :
“There are ties in the preranked stats (0.01% of the list).
The order of those tied genes will be arbitrary, which may produce unexpected results.”


  |======================================================================| 100%

  |======================================================================| 100%



Warning message in preparePathwaysAndStats(pathways, stats, minSize, maxSize, gseaParam, :
“There are ties in the preranked stats (0.01% of the list).
The order of those tied genes will be arbitrary, which may produce unexpected results.”


  |======================================================================| 100%

  |======================================================================| 100%



...

  |======================================================================| 100%



Warning message in preparePathwaysAndStats(pathways, stats, minSize, maxSize, gseaParam, :
“There are ties in the preranked stats (0.01% of the list).
The order of those tied genes will be arbitrary, which may produce unexpected results.”


  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%



Warning message in preparePathwaysAndStats(pathways, stats, minSize, maxSize, gseaParam, :
“There are ties in the preranked stats (0.01% of the list).
The order of those tied genes will be arbitrary, which may produce unexpected results.”


  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |======================================================================| 100%

  |=========================

In [19]:
print('Done!')

[1] "Done!"
